# Section 06: Scaling with Google Cloud Operations

**Companion notebook for**: `06-scaling-operations.html`

## Learning Objectives
- Understand cloud cost management strategies and billing
- Work with the Google Cloud resource hierarchy
- Apply SRE concepts: SLIs, SLOs, SLAs, error budgets
- Explore DevOps practices and DORA metrics
- Use Cloud Monitoring and Logging concepts
- Understand Google's sustainability commitments

## Prerequisites
- A Google Cloud project (optional — mock outputs provided)
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-billing google-cloud-monitoring google-cloud-logging

In [ ]:
import os
import json

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
print(f"Project ID: {PROJECT_ID}")

if PROJECT_ID == "your-project-id":
    print("WARNING: Mock outputs will be used.")

---
## Section 1: Cost Management Strategies

Cloud cost optimization is a continuous process, not a one-time activity.

In [ ]:
# Cost optimization strategies
strategies = [
    {
        "strategy": "Right-sizing",
        "description": "Match VM types to actual workload requirements",
        "how": "Use Cloud Monitoring CPU/memory metrics to identify over-provisioned VMs",
        "savings": "20-50% on compute costs",
    },
    {
        "strategy": "Sustained Use Discounts",
        "description": "Automatic discount for VMs running 25%+ of a month",
        "how": "No action needed — automatically applied to Compute Engine and GKE",
        "savings": "Up to 30%",
    },
    {
        "strategy": "Committed Use Discounts (CUDs)",
        "description": "1 or 3 year commitment for predictable workloads",
        "how": "Purchase commitments for vCPUs and memory (not specific machine types)",
        "savings": "Up to 57% (3-year)",
    },
    {
        "strategy": "Spot / Preemptible VMs",
        "description": "Interruptible VMs at steep discount",
        "how": "Use for batch processing, CI/CD, fault-tolerant workloads",
        "savings": "60-91%",
    },
    {
        "strategy": "Autoscaling",
        "description": "Scale resources based on demand",
        "how": "MIGs for VMs, HPA for GKE, auto-scaling for Cloud Run",
        "savings": "Pay only for peak, not constant peak capacity",
    },
    {
        "strategy": "Serverless",
        "description": "Scale to zero when idle",
        "how": "Cloud Run, Cloud Functions, BigQuery — no idle costs",
        "savings": "100% when no traffic",
    },
    {
        "strategy": "Storage Lifecycle",
        "description": "Transition data to cheaper storage classes",
        "how": "Lifecycle rules: Standard -> Nearline -> Coldline -> Archive -> Delete",
        "savings": "Up to 94% vs Standard",
    },
    {
        "strategy": "Budget Alerts",
        "description": "Get notified when spending exceeds thresholds",
        "how": "Set budgets at 50%, 90%, 100% of target. Alerts are notifications only.",
        "savings": "Prevent surprise bills through awareness",
    },
]

print("Cloud Cost Optimization Strategies")
print("=" * 70)
for s in strategies:
    print(f"\n{s['strategy']}")
    print(f"  What:    {s['description']}")
    print(f"  How:     {s['how']}")
    print(f"  Savings: {s['savings']}")

---
## Section 2: Billing Structure and Tools

Understanding the billing hierarchy is essential for cost governance.

In [ ]:
# Billing hierarchy and tools
print("Google Cloud Billing Hierarchy")
print("=" * 50)
print("""
  [ Billing Account ]
       |
       |-- [ Project A ] -- all resources billed here
       |-- [ Project B ] -- separate billing boundary  
       |-- [ Project C ]
       
  Key concepts:
  - Each project links to exactly ONE billing account
  - A billing account can have MANY projects
  - Labels on resources enable cost allocation
""")

billing_tools = [
    ("Billing Console", "View costs, forecasts, and trends by project/service"),
    ("Budgets & Alerts", "Set thresholds that trigger email/Pub/Sub notifications"),
    ("Billing Export", "Export detailed billing data to BigQuery for analysis"),
    ("Cost Table Report", "Detailed breakdown of costs by SKU"),
    ("Pricing Calculator", "Estimate costs before deploying resources"),
    ("Recommender", "AI-powered suggestions for right-sizing and CUDs"),
    ("Labels", "Key-value tags for cost allocation (e.g., team:frontend)"),
]

print("Billing Tools")
print("-" * 70)
for name, desc in billing_tools:
    print(f"  {name:<22} {desc}")

print("\nCRITICAL EXAM FACT:")
print("  Budget alerts are NOTIFICATIONS ONLY.")
print("  They do NOT stop spending automatically.")
print("  To cap spending: Pub/Sub + Cloud Functions to disable billing.")

---
## Section 3: Resource Hierarchy

Google Cloud organizes resources in a hierarchy that controls
access, policies, and billing.

In [ ]:
# Resource hierarchy
print("Google Cloud Resource Hierarchy")
print("=" * 60)

hierarchy = [
    {
        "level": "Organization",
        "purpose": "Root node, maps to company domain (e.g., example.com)",
        "iam": "Org-level policies apply to EVERYTHING below",
        "example": "org: example.com",
    },
    {
        "level": "Folder",
        "purpose": "Group projects by team, department, or environment",
        "iam": "Folder policies inherit to all child projects",
        "example": "folders: Engineering, Finance, Marketing",
    },
    {
        "level": "Project",
        "purpose": "Resource container and billing boundary",
        "iam": "Most IAM is set at project level",
        "example": "projects: web-prod, ml-training, analytics-dev",
    },
    {
        "level": "Resource",
        "purpose": "Individual GCP service instances",
        "iam": "Some resources support resource-level IAM",
        "example": "VM instances, buckets, BigQuery datasets",
    },
]

for h in hierarchy:
    print(f"\n  {h['level']}")
    print(f"    Purpose: {h['purpose']}")
    print(f"    IAM:     {h['iam']}")
    print(f"    Example: {h['example']}")

print("\nKey rule: IAM policies are ADDITIVE and INHERIT downward.")
print("You CANNOT restrict what was granted at a higher level.")

---
## Section 4: SRE Concepts

Site Reliability Engineering is Google's approach to operations.
SLIs, SLOs, SLAs, and error budgets are core CDL exam topics.

In [ ]:
# SRE concepts with examples
print("SRE Core Concepts")
print("=" * 70)

sre_concepts = [
    {
        "concept": "SLI (Service Level Indicator)",
        "definition": "A measurable metric that reflects service health",
        "examples": [
            "Request latency (p99 < 200ms)",
            "Availability (% successful requests)",
            "Error rate (% failed requests)",
            "Throughput (requests per second)",
        ],
    },
    {
        "concept": "SLO (Service Level Objective)",
        "definition": "Internal target value for an SLI over a time window",
        "examples": [
            "99.9% availability over 30 days",
            "p99 latency < 200ms over 7 days",
            "Error rate < 0.1% over 30 days",
        ],
    },
    {
        "concept": "SLA (Service Level Agreement)",
        "definition": "External contract with business consequences for missing targets",
        "examples": [
            "If availability < 99.9%, customer gets 10% service credit",
            "If availability < 99.0%, customer gets 30% service credit",
            "SLAs are always LESS strict than SLOs",
        ],
    },
    {
        "concept": "Error Budget",
        "definition": "Allowed amount of unreliability = 100% - SLO",
        "examples": [
            "99.9% SLO = 0.1% error budget = 43.8 min/month",
            "99.99% SLO = 0.01% error budget = 4.38 min/month",
            "99.95% SLO = 0.05% error budget = 21.9 min/month",
        ],
    },
]

for s in sre_concepts:
    print(f"\n{s['concept']}")
    print(f"  Definition: {s['definition']}")
    print(f"  Examples:")
    for ex in s['examples']:
        print(f"    - {ex}")

In [ ]:
# Error budget calculator
def calculate_error_budget(slo_pct, period_days=30):
    """Calculate error budget from SLO percentage."""
    error_budget_pct = 100.0 - slo_pct
    total_minutes = period_days * 24 * 60
    allowed_downtime = total_minutes * (error_budget_pct / 100)
    return {
        "slo": f"{slo_pct}%",
        "error_budget_pct": f"{error_budget_pct}%",
        "allowed_downtime_min": round(allowed_downtime, 1),
        "allowed_downtime_hr": round(allowed_downtime / 60, 2),
    }

print("Error Budget Calculator (30-day period)")
print("=" * 55)
print(f"{'SLO':<12} {'Error Budget':<15} {'Downtime (min)':<16} {'Downtime (hr)'}")
print("-" * 55)

for slo in [99.0, 99.5, 99.9, 99.95, 99.99, 99.999]:
    eb = calculate_error_budget(slo)
    print(f"{eb['slo']:<12} {eb['error_budget_pct']:<15} {eb['allowed_downtime_min']:<16} {eb['allowed_downtime_hr']}")

print("\nError budget philosophy:")
print("  Budget healthy -> ship features faster")
print("  Budget depleted -> focus on reliability, freeze deployments")
print("  Error budget = balance between velocity and reliability")

---
## Section 5: DevOps and DORA Metrics

DORA (DevOps Research and Assessment) metrics measure software
delivery performance.

In [ ]:
# DORA metrics
dora_metrics = [
    {
        "metric": "Deployment Frequency",
        "measures": "How often code is deployed to production",
        "elite": "Multiple times per day",
        "low": "Less than once per month",
    },
    {
        "metric": "Lead Time for Changes",
        "measures": "Time from commit to running in production",
        "elite": "Less than 1 hour",
        "low": "More than 6 months",
    },
    {
        "metric": "Change Failure Rate",
        "measures": "% of deployments causing a failure in production",
        "elite": "0-15%",
        "low": "46-60%",
    },
    {
        "metric": "Time to Restore Service",
        "measures": "Time to recover from a production failure",
        "elite": "Less than 1 hour",
        "low": "More than 6 months",
    },
]

print("DORA Metrics — DevOps Performance")
print("=" * 70)
for m in dora_metrics:
    print(f"\n{m['metric']}")
    print(f"  Measures: {m['measures']}")
    print(f"  Elite:    {m['elite']}")
    print(f"  Low:      {m['low']}")

print("\nGoogle Cloud DevOps Tools:")
devops_tools = [
    ("Cloud Build", "Serverless CI/CD, build and test containers"),
    ("Cloud Deploy", "Managed CD to GKE/Cloud Run with approvals"),
    ("Artifact Registry", "Store container images and packages"),
    ("Cloud Source Repos", "Private Git repositories on GCP"),
    ("Error Reporting", "Aggregate and track production errors"),
]
for name, desc in devops_tools:
    print(f"  {name:<22} {desc}")

---
## Section 6: Monitoring and Logging

Google Cloud's operations suite provides observability across
all services.

In [ ]:
# Operations suite services
ops_services = [
    ("Cloud Monitoring", "Metrics & dashboards", "CPU, memory, custom metrics, uptime checks, alerting"),
    ("Cloud Logging", "Log management", "Centralized logs, log-based metrics, export to BQ/Storage"),
    ("Cloud Trace", "Distributed tracing", "Track request latency across microservices"),
    ("Error Reporting", "Error aggregation", "Group errors, link to source, alert on new errors"),
    ("Cloud Profiler", "Production profiling", "CPU/memory profiling with <5% overhead"),
]

print("Google Cloud Operations Suite")
print("=" * 80)
print(f"{'Service':<20} {'Category':<22} {'Capabilities'}")
print("-" * 80)
for name, cat, caps in ops_services:
    print(f"{name:<20} {cat:<22} {caps}")

print("\nMonitoring vs Logging:")
print("  Monitoring = 'What is happening NOW?' (metrics, dashboards, alerts)")
print("  Logging    = 'What happened and WHY?' (log entries, audit trails)")
print("  Trace      = 'Where is the bottleneck?' (request path through services)")

print("\nAlerting Best Practices:")
print("  1. Alert on SYMPTOMS (user impact), not causes (internal metrics)")
print("  2. Use multi-condition policies to reduce false alarms")
print("  3. Set thresholds based on SLOs, not arbitrary values")
print("  4. Route alerts to the right team via appropriate channels")

---
## Section 7: Sustainability

Google's environmental commitments are a CDL exam topic.

In [ ]:
# Google Cloud sustainability facts
sustainability = [
    ("Carbon neutral since", "2007"),
    ("24/7 CFE goal", "2030 — every data center, every hour, carbon-free"),
    ("PUE (Power Usage Effectiveness)", "1.1 average (industry avg: 1.6)"),
    ("Renewable energy matching", "100% since 2017"),
    ("Water positive goal", "2030 — replenish more freshwater than consumed"),
    ("Cloud vs on-prem efficiency", "5-10x less carbon for same workload"),
    ("Custom hardware", "Reduces e-waste through efficient design and reuse"),
]

print("Google Cloud Sustainability Commitments")
print("=" * 65)
for fact, value in sustainability:
    print(f"  {fact:<40} {value}")

print("\nCustomer tools:")
tools = [
    "Carbon Footprint dashboard — view emissions by project/service/region",
    "Region carbon data — choose low-carbon regions (Oregon, Finland, Iowa)",
    "Active Assist — recommendations to reduce idle resources (saves money + carbon)",
    "Unattended project recommender — identify unused projects for cleanup",
]
for t in tools:
    print(f"  - {t}")

print("\nExam tip: Moving to Google Cloud reduces carbon by ~5-10x vs on-prem.")
print("This is a business benefit of cloud migration.")

---
## Summary

In this notebook we covered:

1. **Cost Management** — Right-sizing, SUDs, CUDs, Spot VMs, autoscaling, serverless, lifecycle.
2. **Billing** — Billing accounts, projects, budgets (notifications only!), export to BigQuery.
3. **Resource Hierarchy** — Organization > Folder > Project > Resource. IAM inherits downward.
4. **SRE** — SLIs (metrics), SLOs (targets), SLAs (contracts), error budgets (velocity vs reliability).
5. **DevOps** — DORA metrics: deploy frequency, lead time, failure rate, restore time.
6. **Monitoring** — Cloud Monitoring (metrics), Logging (logs), Trace (latency).
7. **Sustainability** — Carbon neutral since 2007, 24/7 CFE by 2030, 1.1 PUE.

**Congratulations!** You have completed all 6 sections of the Cloud Digital Leader study guide.